# prefect smoke test

Flows, tasks, states and settings. The examples run locally and need no network.

In [ ]:
# Example 1: @task composition inside a @flow.
from prefect import flow, task

@task
def add(a: int, b: int) -> int:
    return a + b

@task
def square(x: int) -> int:
    return x * x

@flow
def pipeline(a: int, b: int) -> int:
    s = add(a, b)
    return square(s)

out = pipeline(3, 4)
print(f"pipeline(3, 4) = {out}")
assert out == 49

In [ ]:
# Example 2: prefect.states — construct + inspect terminal states.
from prefect.states import Completed, Failed, Pending

ok = Completed(message="all good")
bad = Failed(message="boom")
pend = Pending()

print(f"Completed: type={ok.type}, is_completed={ok.is_completed()}, msg={ok.message!r}")
print(f"Failed:    type={bad.type}, is_failed={bad.is_failed()}")
print(f"Pending:   type={pend.type}, is_pending={pend.is_pending()}")

assert ok.is_completed()
assert bad.is_failed()
assert pend.is_pending()

In [ ]:
# Example 3: prefect.settings — read the current Settings object.
from prefect.settings import get_current_settings

settings = get_current_settings()
print(f"settings type: {type(settings).__name__}")

api_url = settings.api.url
home = settings.home
print(f"settings.api.url = {api_url!r}")
print(f"settings.home    = {home}")

assert settings is not None
assert hasattr(settings, "api")
assert hasattr(settings, "home")

In [ ]:
# Example 4: async @flow + @task using top-level await (Jupyter runs in an event loop).
import asyncio
from prefect import flow, task

@task
async def fetch_value(x: int) -> int:
    await asyncio.sleep(0)
    return x * 10

@flow
async def async_pipeline(xs: list[int]) -> list[int]:
    return [await fetch_value(x) for x in xs]

values = await async_pipeline([1, 2, 3])
print(f"async_pipeline([1,2,3]) = {values}")
assert values == [10, 20, 30]